<div class='alert alert-warning'>

SciPy's interactive examples with Jupyterlite are experimental and may not always work as expected. Execution of cells containing imports may result in large downloads (up to 60MB of content for the first import from SciPy). Load times when importing from SciPy may take roughly 10-20 seconds. If you notice any problems, feel free to open an [issue](https://github.com/scipy/scipy/issues/new/choose).

</div>

Generate a sinusoid:


In [ ]:
import numpy as np
f1, f2, fs = 8, 10, 200  # Hz
t = np.linspace(0, 1, fs, endpoint=False)
x = np.sin(2*np.pi*t*f2)
import matplotlib.pyplot as plt
plt.plot(t, x)
plt.axis([0, 1, -1.1, 1.1])
plt.show()

Its discrete Fourier transform has all of its energy in a single frequency
bin:


In [ ]:
from scipy.fft import rfft, rfftfreq
from scipy.signal import czt, czt_points
plt.plot(rfftfreq(fs, 1/fs), abs(rfft(x)))
plt.margins(0, 0.1)
plt.show()

However, if the sinusoid is logarithmically-decaying:


In [ ]:
x = np.exp(-t*f1) * np.sin(2*np.pi*t*f2)
plt.plot(t, x)
plt.axis([0, 1, -1.1, 1.1])
plt.show()

the DFT will have spectral leakage:


In [ ]:
plt.plot(rfftfreq(fs, 1/fs), abs(rfft(x)))
plt.margins(0, 0.1)
plt.show()

While the DFT always samples the z-transform around the unit circle, the
chirp z-transform allows us to sample the Z-transform along any
logarithmic spiral, such as a circle with radius smaller than unity:


In [ ]:
M = fs // 2  # Just positive frequencies, like rfft
a = np.exp(-f1/fs)  # Starting point of the circle, radius < 1
w = np.exp(-1j*np.pi/M)  # "Step size" of circle
points = czt_points(M + 1, w, a)  # M + 1 to include Nyquist
from matplotlib.patches import Arc
fig, ax = plt.subplots()
ax.plot(points.real, points.imag, '.')
ax.add_patch(Arc((0, 0), width=2, height=2, theta1=0,
                 theta2=180, alpha=.3))
ax.set_aspect('equal')
ax.set_xlim(-1.05, 1.05)
plt.show()

With the correct radius, this transforms the decaying sinusoid (and others
with the same decay rate) without spectral leakage:


In [ ]:
z_vals = czt(x, M + 1, w, a)  # Include Nyquist for comparison to rfft
freqs = np.angle(points)*fs/(2*np.pi)  # angle = omega, radius = sigma
plt.plot(freqs, abs(z_vals))
plt.margins(0, 0.1)
plt.show()